# Agent Harness — Advanced
### Concurrency, Failure Modes, Framework Internals, and Evaluation
### Session 1B of the Agentic AI Series — Author: Abhishek

**Runs on:** Ollama (local) or Colab (Hugging Face) — same dual-backend
pattern as every other session. No API key, no cost.

**Prerequisite:** Session 1 (Agent Harness Fundamentals). This session
takes the same ticket-triage harness and asks the questions a senior
engineer or a client's architecture review actually asks: what happens
under concurrent load, what happens when a step fails halfway through, what
does a framework buy you that you didn't already build, and how do you
know — with statistical rigor, not vibes — that the harness is still doing
its job three months into a deployment?

---

## Why Session 1 wasn't enough

Session 1's harness is correct for exactly one thing: one ticket, processed
start to finish, synchronously, with nothing else happening at the same
time, and no assumption that anything could fail partway through. That
description matches almost no real production system.

If you have 3+ years in data science, you already know how to validate a
model offline. This session is about everything **around** the model that
you don't get for free just because the model is good: what happens when
50 tickets land in the same second, what happens when the CRM write
succeeds but the notification doesn't, what a framework like LangGraph is
actually doing when it "handles the loop for you," and how you catch a
harness quietly drifting off-spec the way you'd catch a fraud model's
population shifting.

We will deliberately break things in this notebook. That's the point.


## 0. Setup

In [ ]:
import importlib.util
IN_COLAB = importlib.util.find_spec("google.colab") is not None
BACKEND = "huggingface" if IN_COLAB else "ollama"
print(f"Backend: {BACKEND}")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate pydantic langgraph
else:
    %pip install -q ollama pydantic langgraph


In [ ]:
import asyncio
import json
import random
import time
import warnings
from datetime import datetime, timezone
from enum import Enum
from typing import Literal

from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")

if BACKEND == "huggingface":
    from transformers import pipeline
    import torch

    HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    device = 0 if torch.cuda.is_available() else -1
    _generator = pipeline("text-generation", model=HF_MODEL, device=device)

    def call_model_sync(messages, max_new_tokens=300, temperature=0.3):
        output = _generator(messages, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=temperature > 0)
        return output[0]["generated_text"][-1]["content"]

else:
    import ollama
    OLLAMA_MODEL = "llama3.2:3b"

    def call_model_sync(messages, max_new_tokens=300, temperature=0.3):
        response = ollama.chat(model=OLLAMA_MODEL, messages=messages,
                                options={"num_predict": max_new_tokens, "temperature": temperature})
        return response["message"]["content"]


async def call_model(messages, max_new_tokens=300, temperature=0.3):
    """Async wrapper around a synchronous model call.

    This matters more than it looks: neither the Ollama Python client nor a local
    Hugging Face pipeline is natively async - both block the calling thread for the
    full duration of inference. `asyncio.to_thread` runs the blocking call in a worker
    thread so the rest of your event loop (other tickets' I/O, timeouts, cancellation)
    keeps moving. This is the same pattern you would use to make any blocking C
    extension or blocking SDK call cooperate with asyncio.
    """
    return await asyncio.to_thread(call_model_sync, messages, max_new_tokens, temperature)

print("call_model() is ready (async wrapper over a blocking backend).")


### The concurrency ceiling you actually have

Before writing a single line of concurrent code, know the real constraint:
Ollama's default server processes **one generation request at a time**
unless you explicitly raise `OLLAMA_NUM_PARALLEL`, and a Hugging Face
pipeline on a single GPU has exactly one GPU to share. Concurrency in this
notebook buys you overlap on the **I/O-bound and CPU-bound parts** of the
harness (CRM lookups, guardrail checks, waiting on other tickets) — it does
not multiply your raw model throughput unless you've also scaled the
inference backend. Conflating "my code is concurrent" with "my model calls
are parallelized" is a real mistake worth naming early, especially for a
local-model deployment where GPU memory is the hard limit — the same limit
you benchmarked directly in the CUDA session.


## 1. The baseline harness, condensed

Same domain as Session 1 — ticket triage feeding a CRM — rebuilt async and
condensed so this notebook is self-contained. If you did Session 1, every
piece here should look familiar; only the `async`/`await` is new.


In [ ]:
class Priority(str, Enum):
    LOW = "Low"; MEDIUM = "Medium"; HIGH = "High"; CRITICAL = "Critical"

class Category(str, Enum):
    TECHNICAL = "Technical"; BILLING = "Billing"; ACCOUNT = "Account"; HOW_TO = "How-To"

class TicketTriage(BaseModel):
    priority: Priority
    category: Category
    sentiment: Literal["Positive", "Neutral", "Negative", "Angry"]
    requires_human_review: bool
    reasoning: str


CRM_DATABASE = {
    "cust_1001": {"name": "Rajesh Kumar", "plan": "Enterprise", "open_tickets": 0},
    "cust_1002": {"name": "Priya Sharma", "plan": "Standard", "open_tickets": 2},
    "cust_1003": {"name": "Amit Verma", "plan": "Trial", "open_tickets": 0},
}
CRM_LOCK = asyncio.Lock()  # protects CRM_DATABASE - see Section 2 for why this exists

AUDIT_LOG = []

def _log(action: str, detail: dict):
    AUDIT_LOG.append({"timestamp": datetime.now(timezone.utc).isoformat(), "action": action, "detail": detail})


async def triage_ticket(ticket: dict) -> TicketTriage:
    prompt = f"""Classify this support ticket. Respond with ONLY a JSON object:
{{"priority": "Low|Medium|High|Critical", "category": "Technical|Billing|Account|How-To",
  "sentiment": "Positive|Neutral|Negative|Angry", "requires_human_review": true|false, "reasoning": "one sentence"}}

Subject: {ticket['subject']}
Body: {ticket['body']}

JSON:"""
    for attempt in range(2):
        raw = await call_model([{"role": "user", "content": prompt}], temperature=0.1)
        try:
            start, end = raw.find("{"), raw.rfind("}") + 1
            return TicketTriage(**json.loads(raw[start:end]))
        except Exception:
            if attempt == 0:
                continue
            raise

print("Baseline harness loaded.")


## 2. Concurrency: the race condition your unit tests won't catch

Two tickets for the *same customer* arrive close together. Both handlers
read `open_tickets`, both increment it, both write back. Classic
read-modify-write race — the exact bug class behind more production
incidents than almost anything else in distributed systems, and one that
passes every single-threaded test you might write.


In [ ]:
async def increment_open_tickets_unsafe(customer_id: str):
    """Deliberately racy: read, then (simulate real latency) do work, then write."""
    current = CRM_DATABASE[customer_id]["open_tickets"]
    await asyncio.sleep(0.01)  # simulates a network round-trip to a real CRM - this is where the race window opens
    CRM_DATABASE[customer_id]["open_tickets"] = current + 1


async def demonstrate_race():
    CRM_DATABASE["cust_1002"]["open_tickets"] = 0
    # 10 tickets for the same customer, arriving "simultaneously"
    await asyncio.gather(*[increment_open_tickets_unsafe("cust_1002") for _ in range(10)])
    print(f"Expected open_tickets = 10, got {CRM_DATABASE['cust_1002']['open_tickets']}")

await demonstrate_race()


Run that cell a few times. The count is usually *less* than 10 — some
increments were lost because two coroutines read the same stale value
before either had written back. This is invisible in Session 1's
sequential-only harness, and it is exactly the class of bug a client's
production incident review will find the week after go-live if it isn't
addressed here.

**The fix is the same one you'd reach for in SQL: a lock scoped to the
resource being mutated** — a row-level lock, not a table-level one, or
you'd serialize every ticket regardless of customer and throw away your
concurrency entirely.


In [ ]:
CUSTOMER_LOCKS = {}  # one lock per customer_id, created on first use

def _lock_for(customer_id: str) -> asyncio.Lock:
    if customer_id not in CUSTOMER_LOCKS:
        CUSTOMER_LOCKS[customer_id] = asyncio.Lock()
    return CUSTOMER_LOCKS[customer_id]


async def increment_open_tickets_safe(customer_id: str):
    async with _lock_for(customer_id):
        current = CRM_DATABASE[customer_id]["open_tickets"]
        await asyncio.sleep(0.01)
        CRM_DATABASE[customer_id]["open_tickets"] = current + 1


async def demonstrate_fix():
    CRM_DATABASE["cust_1002"]["open_tickets"] = 0
    await asyncio.gather(*[increment_open_tickets_safe("cust_1002") for _ in range(10)])
    print(f"Expected open_tickets = 10, got {CRM_DATABASE['cust_1002']['open_tickets']}")

await demonstrate_fix()


Notice the lock is **per customer**, not global. Two tickets for *different*
customers should never block each other — that's the difference between a
lock that protects correctness and a lock that silently turns your
concurrent system back into a sequential one. This granularity decision is
exactly the kind of judgment call that separates "it works in the demo"
from "it survives a real load pattern."


## 2b. Bounding concurrency at the model boundary

Given the concurrency ceiling from Section 0, uncapped `asyncio.gather`
across many tickets will queue every model call behind the same
bottleneck, with no backpressure — and if your backend does have some
parallelism (a multi-GPU box, a hosted endpoint with rate limits), an
unbounded fan-out can also just get you throttled or OOM the GPU. A
`Semaphore` caps how many are in flight at once, which is the correct
default posture regardless of backend.


In [ ]:
MODEL_CONCURRENCY_LIMIT = 3
_model_semaphore = asyncio.Semaphore(MODEL_CONCURRENCY_LIMIT)

async def triage_ticket_bounded(ticket: dict) -> TicketTriage:
    async with _model_semaphore:
        return await triage_ticket(ticket)


SAMPLE_TICKETS = [
    {"ticket_id": f"TCK-{5000+i}", "customer_id": "cust_1001",
     "subject": "Issue " + str(i), "body": f"Sample ticket body number {i} describing a problem."}
    for i in range(6)
]

start = time.perf_counter()
results = await asyncio.gather(*[triage_ticket_bounded(t) for t in SAMPLE_TICKETS])
elapsed = time.perf_counter() - start
print(f"Triaged {len(SAMPLE_TICKETS)} tickets with max {MODEL_CONCURRENCY_LIMIT} concurrent model calls in {elapsed:.1f}s")


## 3. Framework internals: what LangGraph actually does for you

Session 1 (and the earlier LangChain/LangGraph track) showed you *how* to
use `StateGraph`. Here's *why* it's structured that way, and what you give
up by hand-rolling the loop yourself: **resumability after a crash.**

A `StateGraph` executes in discrete "super-steps." A checkpointer
serializes the entire state after every super-step. If the process dies
between super-steps 2 and 3, a fresh process can resume from the step-2
checkpoint — it does not need to know anything about *why* the crash
happened. Your hand-rolled `for` loop from Session 1 has no such recovery
point: kill it mid-loop and every bit of progress is gone.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class HarnessState(TypedDict):
    ticket_id: str
    triage: dict
    actions_done: list

def triage_node(state: HarnessState) -> dict:
    # Synchronous stand-in for this demo - the point is the checkpointing behavior, not the model call itself
    return {"triage": {"priority": "High", "category": "Technical"}, "actions_done": []}

def update_priority_node(state: HarnessState) -> dict:
    return {"actions_done": state["actions_done"] + ["update_ticket_priority"]}

def escalate_node(state: HarnessState) -> dict:
    return {"actions_done": state["actions_done"] + ["escalate_to_human"]}

builder = StateGraph(HarnessState)
builder.add_node("triage", triage_node)
builder.add_node("update_priority", update_priority_node)
builder.add_node("escalate", escalate_node)
builder.add_edge(START, "triage")
builder.add_edge("triage", "update_priority")
builder.add_edge("update_priority", "escalate")
builder.add_edge("escalate", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "crash-demo"}}
# Run only as far as "triage" to simulate a crash right after the first super-step
result = None
for step_result in graph.stream({"ticket_id": "TCK-9001", "triage": {}, "actions_done": []}, config, stream_mode="values"):
    result = step_result
    print("Completed super-step, state so far:", result)
    break  # simulate the process dying here, after exactly one super-step


In [ ]:
# "Restart": a brand new call, same thread_id, no in-memory knowledge of what just happened above
snapshot = graph.get_state(config)
print("Recovered state after the simulated crash:", snapshot.values)
print("Next node(s) the graph would run:", snapshot.next)

# Resume - the graph picks up exactly where it left off, no replay of the triage step
final = graph.invoke(None, config)
print("Final state after resuming:", final)


That's the trade you're actually making when you pick a framework over a
hand-rolled loop: **you give up some control over the exact control flow,
in exchange for crash recovery, replayability, and time-travel debugging
that would otherwise be a substantial engineering project in its own
right.** For a harness that only ever runs for a few seconds in a
notebook, that trade doesn't matter. For a harness running as a long-lived
service processing a queue, it's close to non-negotiable.

The OpenAI Agents SDK makes the opposite trade by default: `Runner.run`
gives you the tool loop in one call with a `max_turns` bound, but no
checkpointing out of the box — you get velocity, and you're expected to
add your own persistence layer if you need resumability. Neither choice is
wrong; know which one you're making.


## 4. Idempotency: the bug that survives every retry you add

A `send_refund` call times out. Your retry logic — reasonably — fires
again. But the first call actually *succeeded* on the CRM side before the
timeout; the customer is about to be refunded twice. This is not a
hypothetical: it is the single most common failure mode in any system that
combines retries with a non-idempotent side effect.


In [ ]:
EXECUTED_ACTIONS = {}  # idempotency store: key -> result. In production this is a database row, not a dict.

def _idempotency_key(action: str, ticket_id: str, **kwargs) -> str:
    payload = json.dumps({"action": action, "ticket_id": ticket_id, **kwargs}, sort_keys=True)
    return f"{action}:{ticket_id}:{hash(payload)}"


def send_refund_raw(ticket_id: str, amount: float) -> str:
    return f"Refunded {amount} for {ticket_id}"


def send_refund_idempotent(ticket_id: str, amount: float) -> str:
    key = _idempotency_key("send_refund", ticket_id, amount=amount)
    if key in EXECUTED_ACTIONS:
        _log("idempotent_replay_blocked", {"key": key})
        return EXECUTED_ACTIONS[key]  # return the ORIGINAL result, do not re-execute
    result = send_refund_raw(ticket_id, amount)
    EXECUTED_ACTIONS[key] = result
    return result


# Simulate: the call "succeeds" but the caller times out and retries
first_call = send_refund_idempotent("TCK-4473", 5000)
retry_call = send_refund_idempotent("TCK-4473", 5000)  # the retry - must NOT double-refund
print("First call: ", first_call)
print("Retry call:", retry_call)
print("Total refund actions actually executed:", len([e for e in AUDIT_LOG if e["action"] == "idempotent_replay_blocked"]) == 0 and 1 or "blocked correctly")


### Partial failure across a multi-step action: the saga pattern

A ticket resolution might be: (1) update CRM priority, (2) send a
notification, (3) log the resolution. Step 2 fails. Do you: retry just
step 2? Roll back step 1? Leave it inconsistent and alert a human? This is
a **saga** — a sequence of steps with defined compensating actions for
partial failure, the same pattern you'd use for a multi-step distributed
transaction anywhere else.


In [ ]:
class StepFailure(Exception):
    pass


def flaky_send_notification(ticket_id: str, fail_rate: float = 0.5) -> str:
    if random.random() < fail_rate:
        raise StepFailure(f"Notification service unavailable for {ticket_id}")
    return f"Notification sent for {ticket_id}"


async def retry_with_backoff(fn, *args, max_attempts=3, base_delay=0.2, **kwargs):
    for attempt in range(max_attempts):
        try:
            return fn(*args, **kwargs)
        except StepFailure as e:
            if attempt == max_attempts - 1:
                raise
            delay = base_delay * (2 ** attempt)  # exponential backoff
            _log("retrying_after_failure", {"attempt": attempt + 1, "delay": delay, "error": str(e)})
            await asyncio.sleep(delay)


async def resolve_ticket_saga(ticket_id: str, priority: str):
    completed_steps = []
    try:
        update_ticket_priority_result = f"{ticket_id} priority set to {priority}"
        completed_steps.append(("update_priority", update_ticket_priority_result))
        _log("step_completed", {"step": "update_priority"})

        notify_result = await retry_with_backoff(flaky_send_notification, ticket_id, fail_rate=0.6)
        completed_steps.append(("notify", notify_result))
        _log("step_completed", {"step": "notify"})

        return {"status": "success", "steps": completed_steps}

    except StepFailure as e:
        # Compensating action: the priority update stands (harmless on its own), but we
        # must not silently drop the notification failure - escalate instead of pretending success.
        _log("saga_failed_escalating", {"completed_steps": [s[0] for s in completed_steps], "error": str(e)})
        return {"status": "escalated", "completed_steps": completed_steps, "reason": str(e)}


result = await resolve_ticket_saga("TCK-6001", "High")
print(json.dumps(result, indent=2, default=str))


Run the cell above a few times — the retry/backoff logic means it usually
succeeds, but not always, and when it genuinely can't, **it escalates
loudly instead of silently reporting success.** A harness that reports
"done" when it isn't is worse than one that fails visibly; this is the
production-failure-modes equivalent of a model that's confidently wrong.


### Latency under load: percentiles, not averages

The average latency of your harness is close to useless for capacity
planning. What you actually need to know: what does the *slowest 5% of
requests* look like, because that's what determines whether your SLA holds
under real traffic. This is the same discipline as reporting model
performance with a full distribution instead of a single accuracy number.


In [ ]:
async def timed_triage(ticket):
    start = time.perf_counter()
    await triage_ticket_bounded(ticket)
    return time.perf_counter() - start


LOAD_TEST_TICKETS = [
    {"ticket_id": f"TCK-{7000+i}", "customer_id": "cust_1001", "subject": f"Load test {i}", "body": "Sample body."}
    for i in range(12)
]

latencies = await asyncio.gather(*[timed_triage(t) for t in LOAD_TEST_TICKETS])
latencies_sorted = sorted(latencies)

def percentile(data, p):
    idx = int(len(data) * p / 100)
    return data[min(idx, len(data) - 1)]

print(f"n = {len(latencies)}")
print(f"p50 (median): {percentile(latencies_sorted, 50):.2f}s")
print(f"p90:          {percentile(latencies_sorted, 90):.2f}s")
print(f"p99:          {percentile(latencies_sorted, 99):.2f}s")
print(f"mean:         {sum(latencies) / len(latencies):.2f}s   <- reports this alone and you will miss the tail")


## 5. Evaluation: this is your home turf, raised to harness level

You already know how to build an offline eval set. The shift for an agent
harness: you're evaluating a **pipeline with a stochastic component and
consequential actions**, not a single classifier's predictions. Three
things change: what counts as a false negative, how you detect drift in a
generative component, and how you validate the parts that don't have a
single correct answer.


In [ ]:
class GoldExample(BaseModel):
    ticket: dict
    expected_priority: Priority
    expected_requires_review: bool


GOLD_SET = [
    GoldExample(
        ticket={"ticket_id": "EVAL-1", "customer_id": "cust_1001",
                "subject": "Production down", "body": "All customers are seeing 500 errors right now."},
        expected_priority=Priority.CRITICAL, expected_requires_review=True,
    ),
    GoldExample(
        ticket={"ticket_id": "EVAL-2", "customer_id": "cust_1002",
                "subject": "PDF export question", "body": "How do I export a report as PDF?"},
        expected_priority=Priority.LOW, expected_requires_review=False,
    ),
    GoldExample(
        ticket={"ticket_id": "EVAL-3", "customer_id": "cust_1003",
                "subject": "Cancel and refund", "body": "Cancel my account and refund my last payment immediately."},
        expected_priority=Priority.HIGH, expected_requires_review=True,
    ),
]


async def run_eval(gold_set: list[GoldExample]) -> dict:
    correct_priority = 0
    review_false_negatives = 0  # SHOULD have required review, but didn't - the dangerous direction
    review_false_positives = 0  # flagged for review when it didn't need to be - a cost, not a risk

    for example in gold_set:
        prediction = await triage_ticket(example.ticket)
        if prediction.priority == example.expected_priority:
            correct_priority += 1
        if example.expected_requires_review and not prediction.requires_human_review:
            review_false_negatives += 1
        if not example.expected_requires_review and prediction.requires_human_review:
            review_false_positives += 1

    n = len(gold_set)
    return {
        "n": n,
        "priority_accuracy": correct_priority / n,
        "review_false_negatives": review_false_negatives,  # this number should be a release blocker at > 0
        "review_false_positives": review_false_positives,
    }

eval_result = await run_eval(GOLD_SET)
print(json.dumps(eval_result, indent=2))


**`review_false_negatives` is the metric that matters most, and it's
asymmetric on purpose.** A ticket that gets escalated when it didn't need
to be costs a few minutes of a human's time. A critical outage ticket that
*doesn't* get escalated because the model under-called it is the incident
that ends up in a postmortem. Treat this number the way you'd treat a
false-negative rate on a fraud or safety classifier — as the primary gate,
not a secondary metric.

### Drift: does the harness give the same answer to the same ticket over time?

A generative component means re-running the exact same input can produce a
different output. That's not a bug to eliminate — it's a property to
*monitor*, the same way you'd monitor population stability on a model's
input distribution in production.


In [ ]:
async def measure_decision_stability(ticket: dict, n_runs: int = 5) -> dict:
    """Run the same ticket through triage multiple times and measure how much the
    categorical decision varies. High variance on a ticket that should be unambiguous
    is an early warning sign - of prompt fragility, model degradation after an upgrade,
    or a ticket type your harness genuinely can't triage consistently."""
    priorities = []
    reviews = []
    for _ in range(n_runs):
        result = await triage_ticket(ticket)
        priorities.append(result.priority.value)
        reviews.append(result.requires_human_review)

    from collections import Counter
    priority_counts = Counter(priorities)
    most_common_share = priority_counts.most_common(1)[0][1] / n_runs

    return {
        "n_runs": n_runs,
        "priority_distribution": dict(priority_counts),
        "modal_agreement_rate": most_common_share,  # 1.0 = perfectly stable, lower = concerning
        "review_flag_agreement_rate": max(reviews.count(True), reviews.count(False)) / n_runs,
    }

stability = await measure_decision_stability(GOLD_SET[0].ticket, n_runs=5)
print(json.dumps(stability, indent=2))


A `modal_agreement_rate` below roughly 0.8 on a ticket that a human would
call unambiguous is worth treating as a signal, not noise — either the
prompt needs tightening, the model is a poor fit for this decision
boundary, or (worth considering honestly) the ticket genuinely is
ambiguous and your gold label is wrong. All three are useful things to
learn, and none of them show up if you only ever run the eval set once.

### LLM-as-judge, with the caveat stated up front

Some outputs — `draft_reply`'s tone, `reasoning`'s coherence — don't have a
single correct answer to exact-match against. A second model call can
grade them, but a judge model has the same failure modes as any other
model: position bias, verbosity bias, and a tendency to rate its own
phrasing style highly. Treat judge scores as one weak signal among several,
the same way you'd treat one annotator's label in a low-inter-rater-
reliability labeling task — not as ground truth.


In [ ]:
class JudgeVerdict(BaseModel):
    is_empathetic: bool
    is_appropriate_length: bool
    concerns: str


async def judge_reply(customer_message: str, draft: str) -> JudgeVerdict:
    prompt = f"""Judge this customer support reply. Respond with ONLY JSON:
{{"is_empathetic": true|false, "is_appropriate_length": true|false, "concerns": "brief note or empty string"}}

Customer's message: {customer_message}
Draft reply: {draft}

JSON:"""
    raw = await call_model([{"role": "user", "content": prompt}], temperature=0.0)
    start, end = raw.find("{"), raw.rfind("}") + 1
    return JudgeVerdict(**json.loads(raw[start:end]))


sample_draft = "I'm very sorry to hear about this frustrating experience. We're on it right away."
verdict = await judge_reply("I want to cancel and I'm very unhappy.", sample_draft)
print(verdict.model_dump_json(indent=2))


## 6. A production readiness checklist

Use this as a template for any agent harness you build after this session
— not just this notebook's ticket-triage example.

| Dimension | Question to answer before go-live | Built in this session |
|---|---|---|
| Concurrency safety | Are shared resources locked at the right granularity? | Per-customer `asyncio.Lock` |
| Backpressure | Is model/backend concurrency explicitly bounded? | `asyncio.Semaphore` |
| Resumability | Can the harness recover mid-run after a crash? | LangGraph checkpointing |
| Idempotency | Can a retried action double-execute a side effect? | Idempotency key store |
| Partial failure | Is there a defined compensating action per failure mode? | Saga + backoff retry |
| Latency SLAs | Do you know p90/p99, not just the average? | Percentile load test |
| Eval coverage | Is there a gold set with an asymmetric-risk metric? | `review_false_negatives` |
| Drift monitoring | Do you measure decision stability over time, not just once? | Modal agreement rate |
| Subjective quality | Is judge-model scoring treated as a weak signal, not ground truth? | `judge_reply` with caveats |
| Governance (Session 1) | Audit log, risk-tiered approval gates | Carried forward unchanged |

## Exercises

1. **Compensating rollback.** Extend `resolve_ticket_saga` so that if the
   notification step exhausts all retries, it actively reverts the
   priority update (not just escalates) — implement a real compensating
   action, not just a log line.
2. **A second eval dimension.** Add a gold example where the *correct*
   answer is ambiguous even to a human, and design a metric that doesn't
   penalize the harness for disagreeing with a single gold label on a
   genuinely contested case.
3. **Correlate concurrent runs.** Add a `correlation_id` to every
   `_log()` call so that when 10 tickets run concurrently, you can filter
   the audit log down to exactly one ticket's causal chain — this is the
   distributed-tracing problem in miniature.
